In [1]:
import numpy as np
import pandas as pd

In [2]:
dfpro=pd.read_csv("product.csv")
dfsale=pd.read_csv("sales_data.csv")
dfstore=pd.read_csv("strores.csv")


In [3]:
print(dfpro.shape)
print(dfpro.head(5))

(12, 4)
   product_id  product_name     category  price
0         501        Laptop  Electronics  60000
1         502    Smartphone  Electronics  25000
2         503    Headphones  Electronics   2000
3         504  Office Chair    Furniture   5500
4         505   Study Table    Furniture   7000


In [4]:
print(dfsale.shape)
print(dfsale.head(5))

(15, 6)
   sale_id  store_id  product_id  quantity   sale_date  amount
0     1001       101         501       2.0  01-06-2026  1200.0
1     1002       102         502       1.0  02-06-2026   800.0
2     1003       103         503       NaN  03-06-2026  1500.0
3     1004       101         504       5.0  04-06-2026     NaN
4     1005       104         505       3.0  05-06-2026  2100.0


In [5]:
print(dfstore.shape)
print(dfstore.head(5))

(10, 4)
   store_id       store_name       city region
0       101       Tech World     Jaipur  North
1       102       Smart Mart      Delhi  North
2       103  Home Essentials     Mumbai   West
3       104      Super Store  Bengaluru  South
4       105      City Bazaar  Hyderabad  South


In [6]:
print(dfpro.isnull().sum())

product_id      0
product_name    0
category        0
price           0
dtype: int64


In [7]:
print(dfsale.isnull().sum())
for col in dfsale.columns:
    if dfsale[col].isnull().sum()>0:
        print(col,":",dfsale[col].isnull().sum())

sale_id       0
store_id      0
product_id    0
quantity      2
sale_date     0
amount        3
dtype: int64
quantity : 2
amount : 3


In [8]:
print(dfstore.isnull().sum())

store_id      0
store_name    0
city          0
region        0
dtype: int64


In [9]:
duplicates=dfsale.duplicated().sum()
print("Duplicate was found:",duplicates)
dfsale=dfsale.drop_duplicates()
print("Duplicates removed:",duplicates)

Duplicate was found: 3
Duplicates removed: 3


In [10]:
print(dfpro.duplicated().sum())
print(dfstore.duplicated().sum())

0
0


In [11]:
dfsale["quantity"]=dfsale["quantity"].fillna(0)
dfsale=dfsale.dropna(subset=["amount"])


In [12]:
print(dfsale.isnull().sum())
print(dfsale)

sale_id       0
store_id      0
product_id    0
quantity      0
sale_date     0
amount        0
dtype: int64
    sale_id  store_id  product_id  quantity   sale_date  amount
0      1001       101         501       2.0  01-06-2026  1200.0
1      1002       102         502       1.0  02-06-2026   800.0
2      1003       103         503       0.0  03-06-2026  1500.0
4      1005       104         505       3.0  05-06-2026  2100.0
5      1006       102         506       4.0  06-06-2026  3200.0
6      1007       103         507       0.0  07-06-2026  1800.0
8      1009       104         509       1.0  09-06-2026   950.0
9      1010       102         510       6.0  10-06-2026  4500.0
13     1011       103         511       4.0  11-06-2026  2600.0
14     1012       101         512       2.0  12-06-2026  1700.0


In [13]:
dfsale["sale_date"] = pd.to_datetime(dfsale["sale_date"])
print(dfsale["sale_date"])
dfsale["amount"]=dfsale["amount"].astype(float)

0    2026-01-06
1    2026-02-06
2    2026-03-06
4    2026-05-06
5    2026-06-06
6    2026-07-06
8    2026-09-06
9    2026-10-06
13   2026-11-06
14   2026-12-06
Name: sale_date, dtype: datetime64[ns]


In [14]:
merged=pd.merge(dfsale,dfstore,on="store_id",how="inner")
final_merged=pd.merge(merged,dfpro,on="product_id",how="inner")
final_merged.head(5)

,sale_id,store_id,product_id,quantity,sale_date,amount,store_name,city,region,product_name,category,price
0,1001,101,501,2.0,2026-01-06,1200.0,Tech World,Jaipur,North,Laptop,Electronics,60000
1,1002,102,502,1.0,2026-02-06,800.0,Smart Mart,Delhi,North,Smartphone,Electronics,25000
2,1003,103,503,0.0,2026-03-06,1500.0,Home Essentials,Mumbai,West,Headphones,Electronics,2000
3,1005,104,505,3.0,2026-05-06,2100.0,Super Store,Bengaluru,South,Study Table,Furniture,7000
4,1006,102,506,4.0,2026-06-06,3200.0,Smart Mart,Delhi,North,Water Bottle,Home & Kitchen,500


In [15]:
final_merged["total_revenue"]=final_merged["quantity"]*final_merged["price"]

In [16]:
print("mean_revenue:",np.mean(final_merged["total_revenue"]))
print("max_revenue:",np.max(final_merged["total_revenue"]))
print("min_revenue:",np.min(final_merged["total_revenue"]))

mean_revenue: 26450.0
max_revenue: 120000.0
min_revenue: 0.0


In [17]:
print(final_merged.groupby("city")["total_revenue"].sum().sort_values(ascending=False))

city
Jaipur       126000.0
Delhi         81000.0
Mumbai        32000.0
Bengaluru     25500.0
Name: total_revenue, dtype: float64


In [18]:
import sqlite3

In [19]:
conn=sqlite3.connect("retail.db")

In [20]:
final_merged.to_sql("retail_sales",conn,if_exists="replace",index=False)

10

In [21]:
df=pd.read_sql("select * from retail_sales",conn)
print(df.head())

   sale_id  store_id  product_id  quantity            sale_date  amount  \
0     1001       101         501       2.0  2026-01-06 00:00:00  1200.0   
1     1002       102         502       1.0  2026-02-06 00:00:00   800.0   
2     1003       103         503       0.0  2026-03-06 00:00:00  1500.0   
3     1005       104         505       3.0  2026-05-06 00:00:00  2100.0   
4     1006       102         506       4.0  2026-06-06 00:00:00  3200.0   

        store_name       city region  product_name        category  price  \
0       Tech World     Jaipur  North        Laptop     Electronics  60000   
1       Smart Mart      Delhi  North    Smartphone     Electronics  25000   
2  Home Essentials     Mumbai   West    Headphones     Electronics   2000   
3      Super Store  Bengaluru  South   Study Table       Furniture   7000   
4       Smart Mart      Delhi  North  Water Bottle  Home & Kitchen    500   

   total_revenue  
0       120000.0  
1        25000.0  
2            0.0  
3        2

In [22]:
df2=pd.read_sql("select * from retail_sales where product_id in(select product_id from retail_sales group by product_id order by sum(quantity) desc limit 3)",conn)
print(df2.head())

   sale_id  store_id  product_id  quantity            sale_date  amount  \
0     1006       102         506       4.0  2026-06-06 00:00:00  3200.0   
1     1010       102         510       6.0  2026-10-06 00:00:00  4500.0   
2     1011       103         511       4.0  2026-11-06 00:00:00  2600.0   

        store_name    city region    product_name         category  price  \
0       Smart Mart   Delhi  North    Water Bottle   Home & Kitchen    500   
1       Smart Mart   Delhi  North  Microwave Oven  Home Appliances   9000   
2  Home Essentials  Mumbai   West         Printer      Electronics   8000   

   total_revenue  
0         2000.0  
1        54000.0  
2        32000.0  


In [23]:
df3=pd.read_sql("select sum(total_revenue) as total_revenue_per_day, sale_date, store_id,store_name from retail_sales group by sale_date, store_id,store_name order by store_id,sale_date",conn) 
print(df3)

   total_revenue_per_day            sale_date  store_id       store_name
0               120000.0  2026-01-06 00:00:00       101       Tech World
1                 6000.0  2026-12-06 00:00:00       101       Tech World
2                25000.0  2026-02-06 00:00:00       102       Smart Mart
3                 2000.0  2026-06-06 00:00:00       102       Smart Mart
4                54000.0  2026-10-06 00:00:00       102       Smart Mart
5                    0.0  2026-03-06 00:00:00       103  Home Essentials
6                    0.0  2026-07-06 00:00:00       103  Home Essentials
7                32000.0  2026-11-06 00:00:00       103  Home Essentials
8                21000.0  2026-05-06 00:00:00       104      Super Store
9                 4500.0  2026-09-06 00:00:00       104      Super Store


In [24]:

total_transactions = len(final_merged)
total_revenue = final_merged["total_revenue"].sum()
top_city = final_merged.groupby("city")["total_revenue"].sum().idxmax()
top_product = final_merged.groupby("product_name")["quantity"].sum().idxmax()
print("Summary Report")
print(f"Total Transactions : {total_transactions}")
print(f"Total Revenue      : {total_revenue}")
print(f"Top Selling City   : {top_city}")
print(f"Top Selling Product: {top_product}")

Summary Report
Total Transactions : 10
Total Revenue      : 264500.0
Top Selling City   : Jaipur
Top Selling Product: Microwave Oven


In [29]:
def run_pipeline():
    try:
    
        sales = pd.read_csv("sales_data.csv")
        products = pd.read_csv("product.csv")
        stores = pd.read_csv("strores.csv")
        duplicates = sales.duplicated().sum()
        print(f"Duplicate rows removed: {duplicates}")
        sales = sales.drop_duplicates()

    
        sales["quantity"] = sales["quantity"].fillna(0)

    
        sales = sales.dropna(subset=["amount"])
        sales["sale_date"] = pd.to_datetime(sales["sale_date"])
        sales["amount"] = sales["amount"].astype(float)

    
        merged = pd.merge(sales, products, on="product_id", how="inner")

    
        merged = pd.merge(merged, stores, on="store_id", how="inner")


        merged["total_revenue"] = merged["quantity"] * merged["price"]

    
        print("\nRevenue Statistics")
        print("Mean :", np.mean(merged["total_revenue"]))
        print("Max  :", np.max(merged["total_revenue"]))
        print("Min  :", np.min(merged["total_revenue"]))

    
        city_revenue = (
            merged.groupby("city")["total_revenue"]
            .sum()
            .sort_values(ascending=False))
    

        print("\nRevenue by City:")
        print(city_revenue)

        conn = sqlite3.connect("retail.db")

        merged.to_sql(
            "retail_sales",
            conn,
            if_exists="replace",
            index=False
        )
    except FileNotFoundError as e:
        print("Error: One or more input files were not found.")
        print(e)

    except Exception as e:
        print("An unexpected error occurred.")
        print(e)

    

   

In [30]:
run_pipeline()

Duplicate rows removed: 3

Revenue Statistics
Mean : 26450.0
Max  : 120000.0
Min  : 0.0

Revenue by City:
city
Jaipur       126000.0
Delhi         81000.0
Mumbai        32000.0
Bengaluru     25500.0
Name: total_revenue, dtype: float64
